# 自动化评估

## 加载批次推理结果

In [1]:
from pathlib import Path
from typing_extensions import TypedDict
import json

class InferenceItem(TypedDict):
    model: str
    neutral: str
    output: str
    character: str

BATCH_INFERENCE_FILE = Path("./evaluate/outputs/batch_run_result.jsonl")

batch_inference_items: list[InferenceItem] = []

with open(BATCH_INFERENCE_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()
    for line in lines:
        item:InferenceItem = json.loads(line.strip())

        item["output"] = item["output"].split("</think>")[1].strip() if "</think>" in item["output"] else item["output"]

        batch_inference_items.append(item)

## 风格一致性评估

### 加载模型

In [6]:
from pathlib import Path
from typing import Tuple, Dict, List
from tqdm import tqdm

from transformers import AutoModel, AutoTokenizer
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from tokenizers import Tokenizer

import numpy as np
import torch.nn as nn
import torch
import os

BACKBONE_PATH = str(Path('../Models/chinese-roberta-wwm-ext').resolve())
CHECKPOINT_PATH = Path('outputs/style-classifier')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model definition
class CharacterStyleClassifier(nn.Module):
    def __init__(self, backbone_name: str, embed_dim: int = 768, proj_dim: int = 256, num_roles: int = 6, dropout: float = 0.4):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_name, output_hidden_states=False)
        self.hidden_size = self.backbone.config.hidden_size
        assert self.hidden_size == embed_dim, f"Backbone hidden_size {self.hidden_size} != embed_dim {embed_dim}"

        self.proj = nn.Sequential(
            nn.Linear(self.hidden_size, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.classifier = nn.Linear(proj_dim, num_roles)

    def mean_pooling(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_hidden = torch.sum(last_hidden_state * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_hidden / sum_mask
        return mean_pooled

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        last_hidden = out.last_hidden_state
        sent_emb = self.mean_pooling(last_hidden, attention_mask)
        proj = self.proj(sent_emb)
        logits = self.classifier(proj)
        return logits, proj
    
def load_checkpoint(path: str, device='cpu') -> Tuple[nn.Module, 'Tokenizer', LabelEncoder, Dict[str, np.ndarray]]:
    tokenizer = AutoTokenizer.from_pretrained(path)
    with open(os.path.join(path, 'label_encoder.json'), 'r', encoding='utf-8') as f:
        le_json = json.load(f)
    le = LabelEncoder()
    le.classes_ = np.array(le_json['classes'])

    # use the saved backbone
    # 1. 从 config 获取 hidden_size
    backbone_hidden = AutoModel.from_pretrained(path).config.hidden_size
    # 2. 直接从 `path` 初始化模型，这将加载微调后的 backbone
    model = CharacterStyleClassifier(path, embed_dim=backbone_hidden, proj_dim=256, num_roles=len(le.classes_))
    # 3. 加载 head 权重
    chk = torch.load(os.path.join(path, 'head.pt'), map_location=device)
    model.proj.load_state_dict(chk['proj_state'])
    model.classifier.load_state_dict(chk['classifier_state'])
    # 4. 加载 centers
    centers_npz = np.load(os.path.join(path, 'role_centers.npz'))
    role_centers = {k: centers_npz[k] for k in centers_npz.files}
    return model.to(device), tokenizer, le, role_centers

@torch.no_grad()
def compute_role_centers(model: nn.Module, dataloader: DataLoader, label_encoder: LabelEncoder, device='cpu') -> Dict[str, np.ndarray]:
    model.eval()
    accum: Dict[int, List[np.ndarray]] = {}
    for batch in tqdm(dataloader, desc="Computing centers"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].cpu().numpy()
        _, embeddings = model(input_ids=input_ids, attention_mask=attention_mask)
        emb_np = embeddings.detach().cpu().numpy()
        for lbl, e in zip(labels, emb_np):
            accum.setdefault(int(lbl), []).append(e)
    role_centers = {}
    for lbl, vecs in accum.items():
        avg = np.mean(np.stack(vecs, axis=0), axis=0)
        role_name = label_encoder.inverse_transform([lbl])[0]
        role_centers[role_name] = avg
    return role_centers


def get_style_score(model: nn.Module, tokenizer, text: str, role_center: np.ndarray, device='cpu', max_length=128) -> float:
    model.eval()
    enc = tokenizer(text, truncation=True, max_length=max_length, padding='max_length', return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    with torch.no_grad():
        _, emb = model(input_ids=input_ids, attention_mask=attention_mask)
        emb_np = emb.detach().cpu().numpy()[0]
    num = float(np.dot(emb_np, role_center))
    den = float(np.linalg.norm(emb_np) * np.linalg.norm(role_center) + 1e-9)
    return num / den

model, tokenizer, label_encoder, role_centers = load_checkpoint(str(CHECKPOINT_PATH), device)

### 执行评估

In [7]:
import pandas as pd

# Calculate style scores for each item
results = []

for item in tqdm(batch_inference_items, desc="Evaluating Style"):
    character = item['character']
    output_text = item['output']
    
    if not character in role_centers:
        print(f"Warning: Character '{character}' not found in role centers. Skipping.")
        continue

    center = role_centers[character]
    score = get_style_score(model, tokenizer, output_text, center, device=device)
    
    results.append({
        **item,
        "style_score": score
    })
        

# Calculate average score per model
model_scores = {}
for res in results:
    m_name = res['model']
    if m_name not in model_scores:
        model_scores[m_name] = []
    model_scores[m_name].append(res['style_score'])

print("\nStyle Consistency Scores:")
for m_name, scores in model_scores.items():
    avg_score = np.mean(scores)
    print(f"Model: {m_name}, Average Style Score: {avg_score:.4f} (n={len(scores)})")

# Save detailed results
df_results = pd.DataFrame(results)

Evaluating Style: 100%|██████████| 3750/3750 [00:33<00:00, 111.59it/s]


Style Consistency Scores:
Model: Modelv1, Average Style Score: 0.5819 (n=750)
Model: Modelv2, Average Style Score: 0.5778 (n=750)
Model: BaselineA, Average Style Score: 0.8808 (n=750)
Model: BaselineB, Average Style Score: 0.7573 (n=750)
Model: BaselineC, Average Style Score: 0.6870 (n=750)


## 语义保持度评估

### 加载模型

In [4]:
from sentence_transformers import SentenceTransformer, util

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

MODEL_PATH = "../Models/bge-large-zh-v1.5"
semantic_model = SentenceTransformer(MODEL_PATH)

def compute_semantic_similarity(neutral_text, stylized_text):
    """
    计算两个句子的语义相似度 (0 ~ 1)
    """
    # BGE 建议在作为 Query 时添加指令，但作为对称相似度对比，直接编码通常效果已足够好
    # 或者是为两边都添加 "为这个句子生成表示以用于检索相关文章：" (视具体版本文档而定)
    # 对于 STS (Semantic Textual Similarity) 任务，v1.5 通常可以直接使用。
    
    embeddings = semantic_model.encode([neutral_text, stylized_text], normalize_embeddings=True)
    
    # 计算余弦相似度
    similarity = util.cos_sim(embeddings[0], embeddings[1])
    return similarity.item()

### 执行评估

In [5]:
# Calculate semantic scores for each item
for item in tqdm(results, desc="Evaluating Semantic"):
    neutral_text = item['neutral']
    stylized_text = item['output']
    
    # Compute similarity
    score = compute_semantic_similarity(neutral_text, stylized_text)
    
    # Store result
    item['semantic_score'] = score

# Aggregate results
model_metrics = {}

for item in results:
    m_name = item['model']
    char_name = item['character']
    score = item['semantic_score']
    
    if m_name not in model_metrics:
        model_metrics[m_name] = {'total': [], 'per_char': {}}
    
    # Add to total model scores
    model_metrics[m_name]['total'].append(score)
    
    # Add to character specific scores
    if char_name not in model_metrics[m_name]['per_char']:
        model_metrics[m_name]['per_char'][char_name] = []
    model_metrics[m_name]['per_char'][char_name].append(score)

# Print Report
print("\nSemantic Preservation Scores:")
print("=" * 60)
for m_name, metrics in model_metrics.items():
    avg_total = np.mean(metrics['total'])
    print(f"Model: {m_name}")
    
    # Print per-character scores first
    sorted_chars = sorted(metrics['per_char'].keys())
    for char_name in sorted_chars:
        scores = metrics['per_char'][char_name]
        avg_char = np.mean(scores)
        print(f"  - Character: {char_name:<10} | Score: {avg_char:.4f} (n={len(scores)})")
    
    # Print overall model score
    print(f"  >> Overall Average: {avg_total:.4f}")
    print("-" * 60)

# Update the DataFrame with the new semantic scores
df_results = pd.DataFrame(results)

Evaluating Semantic: 100%|██████████| 3750/3750 [02:05<00:00, 29.82it/s]


Semantic Preservation Scores:
Model: Modelv1
  - Character: 凉宫春日       | Score: 0.8145 (n=150)
  - Character: 沐雪         | Score: 0.8348 (n=150)
  - Character: 神里绫华       | Score: 0.8323 (n=150)
  - Character: 胡桃         | Score: 0.8428 (n=150)
  - Character: 钟离         | Score: 0.8321 (n=150)
  >> Overall Average: 0.8313
------------------------------------------------------------
Model: Modelv2
  - Character: 凉宫春日       | Score: 0.8113 (n=150)
  - Character: 沐雪         | Score: 0.8509 (n=150)
  - Character: 神里绫华       | Score: 0.8328 (n=150)
  - Character: 胡桃         | Score: 0.8391 (n=150)
  - Character: 钟离         | Score: 0.8467 (n=150)
  >> Overall Average: 0.8361
------------------------------------------------------------
Model: BaselineA
  - Character: 凉宫春日       | Score: 0.5241 (n=150)
  - Character: 沐雪         | Score: 0.5471 (n=150)
  - Character: 神里绫华       | Score: 0.4831 (n=150)
  - Character: 胡桃         | Score: 0.5233 (n=150)
  - Character: 钟离         | Score: 0.4627 